In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
files = os.listdir(
    "/content/drive/MyDrive/Colab Notebooks/output/statistics"
)
files

['sampled_all_systems.csv',
 'mistral.csv',
 'concat_flaml.csv',
 'llama.csv',
 'concat_ag.csv',
 'concat_rf.csv',
 'hpo_all_systems.csv',
 'result_equal_monotonic_rf_equal.csv',
 'result_monotonic_rf_equal.csv',
 'result_equal_monotonic_flaml_equal.csv',
 'result_monotonic_flaml_equal.csv',
 'result_equal_monotonic_ag_equal.csv',
 'result_monotonic_ag_equal.csv',
 'result_monotonic_llama.csv']

## Mistral

In [4]:
system = "mistral"

In [5]:
df = pd.read_csv(
    f"/content/drive/MyDrive/Colab Notebooks/output/statistics/{system}.csv",
    sep=";"
)
df

,predicted_dataset,selected_llm,number_of_examples,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
0,ed_01,MISTRAL_NEMO_INSTRUCT_2407,0,set(),"{'neutral', 'sarcasm'}",False,0.307054,0.481328,"['sadness', 'joy', 'surprise', 'fear', 'anger'...","['sadness', 'sarcasm', 'joy', 'surprise', 'fea..."
1,ed_01,MISTRAL_NEMO_INSTRUCT_2407,5,set(),set(),False,0.307054,0.543568,"['sadness', 'joy', 'surprise', 'fear', 'anger'...","['sadness', 'joy', 'surprise', 'fear', 'disgus..."
2,ed_02,MISTRAL_NEMO_INSTRUCT_2407,5,set(),"{'fatigue', 'supportive', 'unsure', 'restlessn...",False,0.342871,0.577312,"['sadness', 'joy', 'surprise', 'fear', 'love',...","['fatigue', 'supportive', 'unsure', 'restlessn..."
3,ed_02,MISTRAL_NEMO_INSTRUCT_2407,0,set(),"{'pain', 'angry', 'b', 'shy', 'calm', 'envy', ...",False,0.342871,0.602581,"['sadness', 'joy', 'surprise', 'fear', 'love',...","['pain', 'sadness', 'angry', 'b', 'shy', 'calm..."
4,ed_03,MISTRAL_NEMO_INSTRUCT_2407,5,set(),"{'pain', 'fatigue', 'gratuity', 'determination...",False,0.294867,0.213996,"['sadness', 'embarrassment', 'excitement', 'an...","['fatigue', 'excitement', 'grateitude', 'self-..."
...,...,...,...,...,...,...,...,...,...,...
91,ua_04,MISTRAL_NEMO_INSTRUCT_2407,5,set(),"{'complicated', 'pointy', 'tiny', 'sound', '2'...",False,0.200040,0.271154,"['3', '5', '10', '1', '7']","['complicated', 'pointy', 'tiny', 'sound', '2'..."
92,ua_05,MISTRAL_NEMO_INSTRUCT_2407,0,set(),set(),False,0.776119,0.129353,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
93,ua_05,MISTRAL_NEMO_INSTRUCT_2407,5,set(),set(),False,0.776119,0.124378,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
94,ua_06,MISTRAL_NEMO_INSTRUCT_2407,5,set(),"{'---', '4', 'good'}",False,0.442095,0.128985,"['3', '5', '10', '1', '7']","['3', '---', '4', '5', '10', '1', 'good', '7']"


In [6]:
# Looking at one dataset example
df.query("predicted_dataset == 'ua_01'")

,predicted_dataset,selected_llm,number_of_examples,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
84,ua_01,MISTRAL_NEMO_INSTRUCT_2407,0,set(),"{'---', 'nan'}",False,0.55838,0.245959,"['3', '5', '10', '1', '7']","['3', '---', '5', 'nan', '10', '1', '7']"
85,ua_01,MISTRAL_NEMO_INSTRUCT_2407,5,set(),"{'---', '4', 'nan', 'd', '2', 'deleted', '0'}",False,0.55838,0.324316,"['3', '5', '10', '1', '7']","['3', '---', '4', '5', 'nan', '2', 'd', '10', ..."


In [7]:
grouped_df = (
    df.groupby(
        [
            "predicted_dataset",
            "selected_llm",
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_df = grouped_df.reset_index()
grouped_df

,predicted_dataset,selected_llm,hold_out_f1_score
0,ed_01,MISTRAL_NEMO_INSTRUCT_2407,0.512448
1,ed_02,MISTRAL_NEMO_INSTRUCT_2407,0.589946
2,ed_03,MISTRAL_NEMO_INSTRUCT_2407,0.247046
3,ed_04,MISTRAL_NEMO_INSTRUCT_2407,0.245070
4,ed_05,MISTRAL_NEMO_INSTRUCT_2407,0.391667
5,fn_01,MISTRAL_NEMO_INSTRUCT_2407,0.238605
6,fn_02,MISTRAL_NEMO_INSTRUCT_2407,0.569444
7,fn_03,MISTRAL_NEMO_INSTRUCT_2407,0.231606
8,fn_04,MISTRAL_NEMO_INSTRUCT_2407,0.264744
9,fn_05,MISTRAL_NEMO_INSTRUCT_2407,0.385104


### Result

In [8]:
monotonic = (
    df.sort_values(
        [
            "predicted_dataset",
            "selected_llm",
            "number_of_examples",
        ]
    )
    .groupby(
        [
            "predicted_dataset",
            "selected_llm",
        ],
        as_index=False,
        observed=True
    )
    ["hold_out_f1_score"]
    .apply(lambda x: x.is_monotonic_increasing)
    .rename(columns={"hold_out_f1_score": "is_monotonic_increasing"})
    [["predicted_dataset", "selected_llm", "is_monotonic_increasing"]]
)
result = (
    monotonic.merge(
        df,
        on=["predicted_dataset", "selected_llm"],
    )
    .sort_values(["predicted_dataset", "selected_llm"])
)

result.to_csv(
    f"/content/drive/MyDrive/Colab Notebooks/output/statistics/result_monotonic_{system}.csv",
    sep=";"
)
result

,predicted_dataset,selected_llm,is_monotonic_increasing,number_of_examples,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
0,ed_01,MISTRAL_NEMO_INSTRUCT_2407,True,0,set(),"{'neutral', 'sarcasm'}",False,0.307054,0.481328,"['sadness', 'joy', 'surprise', 'fear', 'anger'...","['sadness', 'sarcasm', 'joy', 'surprise', 'fea..."
1,ed_01,MISTRAL_NEMO_INSTRUCT_2407,True,5,set(),set(),False,0.307054,0.543568,"['sadness', 'joy', 'surprise', 'fear', 'anger'...","['sadness', 'joy', 'surprise', 'fear', 'disgus..."
2,ed_02,MISTRAL_NEMO_INSTRUCT_2407,False,5,set(),"{'fatigue', 'supportive', 'unsure', 'restlessn...",False,0.342871,0.577312,"['sadness', 'joy', 'surprise', 'fear', 'love',...","['fatigue', 'supportive', 'unsure', 'restlessn..."
3,ed_02,MISTRAL_NEMO_INSTRUCT_2407,False,0,set(),"{'pain', 'angry', 'b', 'shy', 'calm', 'envy', ...",False,0.342871,0.602581,"['sadness', 'joy', 'surprise', 'fear', 'love',...","['pain', 'sadness', 'angry', 'b', 'shy', 'calm..."
4,ed_03,MISTRAL_NEMO_INSTRUCT_2407,False,5,set(),"{'pain', 'fatigue', 'gratuity', 'determination...",False,0.294867,0.213996,"['sadness', 'embarrassment', 'excitement', 'an...","['fatigue', 'excitement', 'grateitude', 'self-..."
...,...,...,...,...,...,...,...,...,...,...,...
91,ua_04,MISTRAL_NEMO_INSTRUCT_2407,False,5,set(),"{'complicated', 'pointy', 'tiny', 'sound', '2'...",False,0.200040,0.271154,"['3', '5', '10', '1', '7']","['complicated', 'pointy', 'tiny', 'sound', '2'..."
92,ua_05,MISTRAL_NEMO_INSTRUCT_2407,False,0,set(),set(),False,0.776119,0.129353,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
93,ua_05,MISTRAL_NEMO_INSTRUCT_2407,False,5,set(),set(),False,0.776119,0.124378,"['3', '5', '10', '1', '7']","['3', '5', '10', '1', '7']"
94,ua_06,MISTRAL_NEMO_INSTRUCT_2407,False,5,set(),"{'---', '4', 'good'}",False,0.442095,0.128985,"['3', '5', '10', '1', '7']","['3', '---', '4', '5', '10', '1', 'good', '7']"


# Analysis Examples

Some examples of interesting analysis below:


## Monotonic Analysis

### Monotonic increasing count

In [9]:
is_monotonic_increasing_equal = result["is_monotonic_increasing"]

monotonic_increasing_equal = result.loc[is_monotonic_increasing_equal]
non_monotonic_increasing_equal = result.loc[~is_monotonic_increasing_equal]

print(f"Number EQUAL Monotonic increasing: {len(monotonic_increasing_equal.drop_duplicates('predicted_dataset'))}")
print(f"{monotonic_increasing_equal['predicted_dataset'].drop_duplicates().to_list()}")

print(f"Number EQUAL Non-monotonic increasing: {len(non_monotonic_increasing_equal.drop_duplicates('predicted_dataset'))}")
print(f"{non_monotonic_increasing_equal['predicted_dataset'].drop_duplicates().to_list()}")

Number EQUAL Monotonic increasing: 24
['ed_01', 'fn_02', 'fn_03', 'fn_04', 'fn_05', 'fn_06', 'fn_08', 'fn_09', 'fn_10', 'hs_01', 'hs_02', 'hs_05', 'hs_06', 'hs_07', 'hs_10', 'hs_11', 'hs_13', 'hs_15', 'pc_09', 'sd_01', 'sd_03', 'sd_04', 'ua_01', 'ua_02']
Number EQUAL Non-monotonic increasing: 24
['ed_02', 'ed_03', 'ed_04', 'ed_05', 'fn_01', 'fn_07', 'hs_03', 'hs_04', 'hs_08', 'hs_09', 'hs_12', 'hs_14', 'pc_01', 'pc_02', 'pc_03', 'pc_04', 'pc_05', 'pc_06', 'pc_07', 'pc_08', 'ua_03', 'ua_04', 'ua_05', 'ua_06']


### 0 and 5 hold_out_f1_score difference

In [10]:
da_analysis = (
    result.query("number_of_examples == 0")
    .merge(
        result.query("number_of_examples == 5"),
        on=["selected_llm", "predicted_dataset"],
        suffixes=["_0", "_5"]
    )
)
da_analysis["diff"] = da_analysis["hold_out_f1_score_5"] - da_analysis["hold_out_f1_score_0"]

(
    da_analysis
    .sort_values(
        "predicted_dataset",
        ignore_index=True
    )
    [
        [
            "predicted_dataset",
            "selected_llm",
            "number_of_examples_0",
            "number_of_examples_5",
            "is_monotonic_increasing_0",
            "hold_out_f1_score_0",
            "hold_out_f1_score_5",
            "diff"
        ]
    ]
)


,predicted_dataset,selected_llm,number_of_examples_0,number_of_examples_5,is_monotonic_increasing_0,hold_out_f1_score_0,hold_out_f1_score_5,diff
0,ed_01,MISTRAL_NEMO_INSTRUCT_2407,0,5,True,0.481328,0.543568,0.062241
1,ed_02,MISTRAL_NEMO_INSTRUCT_2407,0,5,False,0.602581,0.577312,-0.025269
2,ed_03,MISTRAL_NEMO_INSTRUCT_2407,0,5,False,0.280096,0.213996,-0.066100
3,ed_04,MISTRAL_NEMO_INSTRUCT_2407,0,5,False,0.297598,0.192542,-0.105056
4,ed_05,MISTRAL_NEMO_INSTRUCT_2407,0,5,False,0.400000,0.383333,-0.016667
5,fn_01,MISTRAL_NEMO_INSTRUCT_2407,0,5,False,0.241138,0.236073,-0.005064
6,fn_02,MISTRAL_NEMO_INSTRUCT_2407,0,5,True,0.569444,0.569444,0.000000
7,fn_03,MISTRAL_NEMO_INSTRUCT_2407,0,5,True,0.219582,0.243630,0.024048
8,fn_04,MISTRAL_NEMO_INSTRUCT_2407,0,5,True,0.250328,0.279161,0.028834
9,fn_05,MISTRAL_NEMO_INSTRUCT_2407,0,5,True,0.373239,0.396969,0.023729
